# 10장. 분류 분석으로 주문 취소 여부 예측하기

이번 과제는 높은 Accuracy 자체보다 **타깃 정의 → 예측 시점 → 누수 방지 → Validation에서 모델/Threshold 선택 → Final Test 독립성 → FP/FN 해석 → Privacy-safe Output**의 순서를 지키는 것이 핵심입니다.

- `completed = 0`
- `cancelled = 1`
- `refunded` 및 기타 상태는 모델링 대상에서 제외
- 모델과 Threshold는 Validation에서 선택
- Final Test는 모든 선택을 고정한 뒤 마지막 평가에만 사용

## 0. 실행 전 확인

Chapter 10은 Chapter 05 전용 오류 데이터가 아니라 공통 프로젝트 `data/raw` → `data/processed` 흐름을 사용합니다.

필요하면 프로젝트 루트에서 다음을 실행합니다.

```powershell
python scripts/prepare_ch10_data.py
```

이 Notebook은 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
ASSIGNMENT_DIR = PROJECT_ROOT / "assignments" / "chapter10"
IMAGE_DIR = ASSIGNMENT_DIR / "images"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python:", sys.executable)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("IMAGE_DIR:", IMAGE_DIR)

## 1. 분류 문제 정의

- **타깃:** 주문 취소 여부 `is_cancelled`
- **0:** completed
- **1:** cancelled
- **제외:** refunded 및 기타 상태
- **예측 단위:** 주문 1건
- **예측 시점:** 주문 생성 직후라고 가정
- **사용 가능 정보:** 주문 생성 시점의 고객 비식별 특성, 결제수단, 주문 상세를 주문 단위로 집계한 수량·금액 정보, 주문 날짜에서 만든 파생 정보
- **제외 정보:** `order_status`, `is_cancelled`, 각종 ID, 취소 사유·취소 시각처럼 결과 이후에 생기는 정보

### 나의 해석과 판단

주문 취소 여부를 예측하면서 이미 확정된 주문 상태나 취소 이후 정보를 입력하면 정답을 미리 알려주는 데이터 누수가 발생합니다. 따라서 실제 운영 시점을 주문 생성 직후로 고정하고, 그 시점에 알 수 있다고 가정한 정보만 사용합니다.

In [ ]:
from src.classification import (
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_classification_dataset,
    build_feature_audit,
    build_split_summary,
    confusion_matrix_dataframe,
    create_prediction_result,
    final_test_evaluation,
    load_classification_source_data,
    public_prediction_result,
    run_classification_analysis,
    select_validation_model,
    split_train_validation_test,
    target_distribution,
    threshold_metrics,
    choose_threshold,
    train_and_compare_on_validation,
    build_classification_validation,
)

data = load_classification_source_data(PROCESSED_DIR)
customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]

for name, frame in data.items():
    print(name, frame.shape)
print("\n주문 상태:")
display(orders["order_status"].value_counts(dropna=False).rename_axis("order_status").reset_index(name="count"))

## 2. Feature Contract와 Leakage Audit

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
cancel_reason
cancelled_at
```

파일에 컬럼이 존재한다는 이유만으로 모델에 넣을 수 있는 것은 아닙니다. **예측 시점보다 뒤에 생기는 정보와 식별자, 타깃 자체**는 모델 입력에서 제외해야 합니다.

In [ ]:
(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

feature_audit = build_feature_audit(numeric_features, categorical_features)

print("숫자형 Feature:", numeric_features)
print("범주형 Feature:", categorical_features)
print("모델링 데이터:", model_data.shape)
display(feature_audit)

selected_features = numeric_features + categorical_features
leakage_overlap = sorted(set(selected_features) & FORBIDDEN_FEATURES)
assert not leakage_overlap
print("Leakage Audit: PASS")

## 3. 주문 단위 특징과 병합 검증

예측 단위가 주문 1건이므로 주문 상세를 주문 단위로 집계합니다.

- `item_count`
- `total_quantity`
- `order_amount`

먼저 `line_total = quantity × unit_price` 관계를 확인한 뒤 집계합니다. 병합은 주문 ↔ 주문특징은 **one_to_one**, 주문 ↔ 고객은 **many_to_one** 관계를 기대합니다.

In [ ]:
items_check = order_items.copy()
expected_line_total = (
    pd.to_numeric(items_check["quantity"], errors="coerce")
    * pd.to_numeric(items_check["unit_price"], errors="coerce")
)
if "line_total" in items_check.columns:
    actual_line_total = pd.to_numeric(items_check["line_total"], errors="coerce")
    line_total_mismatch_count = int((actual_line_total - expected_line_total).abs().gt(1e-6).sum())
else:
    line_total_mismatch_count = 0

order_feature_duplicate_count = int(model_data["order_id"].duplicated().sum())

print("line_total 계산 불일치:", line_total_mismatch_count)
print("모델 데이터 order_id 중복:", order_feature_duplicate_count)
display(merge_checks)
display(data_quality_checks)

assert line_total_mismatch_count == 0
assert order_feature_duplicate_count == 0
assert merge_checks["status"].eq("PASS").all()
print("주문 단위 특징 / 병합 검증: PASS")

### 추가 데이터 품질 확인\n\n공통 Raw 데이터에는 일부 주문에서 `signup_date > order_date`인 시간 모순이 있습니다. 이 경우 원본 날짜를 임의로 수정하거나 해당 주문을 삭제하지 않고, **`days_since_signup` 파생값만 결측으로 표시**합니다. 이후 숫자형 Pipeline의 median imputation은 Train 데이터에서만 학습됩니다. 실제 운영 데이터라면 원천 시스템에서 가입일/주문일을 먼저 정정해야 합니다.\n

### 결과 관찰 및 판단

미매칭이 발생했는데 원인을 확인하지 않고 `fillna(0)`으로 채우면 **실제로 연결이 끊어진 데이터 문제를 정상적인 0 값처럼 위장**하게 됩니다. 따라서 행 수 보존과 미매칭 건수를 먼저 검증하고, 문제가 있으면 모델 학습보다 데이터 관계를 먼저 수정해야 합니다.

## 4. 클래스 비율과 Dummy Baseline

먼저 `completed`와 `cancelled`의 비율을 확인합니다. 클래스가 불균형하면 가장 많은 클래스만 예측해도 Accuracy가 높게 나올 수 있으므로 **Precision, Recall, F1을 함께 봐야 합니다.**

In [ ]:
target_dist = target_distribution(model_data)
display(target_dist)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(target_dist["class_label"], target_dist["count"])
ax.set_title("Chapter 10 Target Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
fig.tight_layout()
fig.savefig(IMAGE_DIR / "step02_class_ratio.png", dpi=150)
plt.show()

## 5. Train / Validation / Test 역할

| Split | 역할 |
|---|---|
| Train | 파라미터 학습 |
| Validation | 후보 모델 선택 + Threshold 선택 |
| Test | 선택 완료 후 마지막 평가 |

이번 실습은 교육용으로 stratified random split을 사용합니다. 실제 미래 주문을 예측하는 운영 환경에서는 **시간 순서 기반 out-of-time 평가**를 추가해야 합니다.

Test를 모델이나 Threshold 선택에 사용하면 Test에 맞춘 선택이 되어 최종 평가의 독립성이 깨집니다.

In [ ]:
(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)
display(split_summary)
print("Train:", X_train.shape, "Validation:", X_valid.shape, "Final Test:", X_test.shape)

## 6. Pipeline 전처리와 Validation 모델 비교

- 숫자형: median imputation → StandardScaler
- 범주형: most_frequent imputation → OneHotEncoder
- 전처리는 Pipeline 안에서 **Train으로만 fit**
- 후보: Dummy Most Frequent, Logistic Regression, Random Forest

Validation의 **F1 → Recall → Precision** 순으로 비베이스라인 모델을 고정합니다.

In [ ]:
(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

display(validation_comparison)

selected_model_name = select_validation_model(validation_comparison)
selected_model = models[selected_model_name]
print("Validation에서 고정한 모델:", selected_model_name)

plot_df = validation_comparison.set_index("model")[["accuracy", "precision", "recall", "f1"]]
fig, ax = plt.subplots(figsize=(9, 5))
plot_df.plot(kind="bar", ax=ax)
ax.set_title("Validation Model Comparison")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "step03_baseline.png", dpi=150)
plt.show()

### 결과 관찰 및 판단

Dummy baseline은 다수 클래스만 예측하기 때문에 Accuracy만 보면 그럴듯해 보일 수 있지만 취소 주문 Recall과 F1이 낮을 수 있습니다. 실제 후보 모델은 **취소 주문을 어느 정도 찾아내면서 Precision과 Recall의 균형을 만드는지**를 기준으로 판단합니다.

## 7. Validation Threshold 선택

Threshold를 낮추면 취소 예측이 늘어 **Recall이 올라갈 수 있지만 FP도 늘 수 있고**, 높이면 반대로 Precision이 올라가면서 FN이 늘 수 있습니다.

Threshold 역시 Final Test를 보지 않고 Validation에서 고정합니다.

In [ ]:
validation_proba = validation_probabilities[selected_model_name]
threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print("선택 모델:", selected_model_name)
print("선택 Threshold:", selected_threshold)
display(threshold_df.sort_values(["f1", "recall", "precision"], ascending=False).head(10))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="Precision")
ax.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="Recall")
ax.plot(threshold_df["threshold"], threshold_df["f1"], marker="o", label="F1")
ax.axvline(selected_threshold, linestyle="--", label=f"Selected={selected_threshold:.2f}")
ax.set_title("Validation Threshold Comparison")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
fig.savefig(IMAGE_DIR / "step05_threshold.png", dpi=150)
plt.show()

### 선택 이유

이번 실습의 자동 선택 규칙은 Validation의 **F1 → Recall → Precision → 더 낮은 Threshold** 순입니다. 실제 업무에서 FP와 FN의 비용이 확인된다면 이 규칙은 업무 목적에 맞게 다시 정의해야 합니다.

## 8. Final Test와 FP/FN 해석

Validation에서 **모델과 Threshold를 이미 고정한 상태**에서 Final Test를 한 번 평가합니다.

- FP: 실제 완료 주문을 취소 위험으로 잘못 분류
- FN: 실제 취소 주문을 완료로 잘못 분류

현재 데이터에는 FP/FN의 실제 금전 비용이 없으므로 어느 오류가 절대적으로 더 비싼지는 확정할 수 없습니다. 다만 취소 위험 선별이 목적이라면 FN을 줄이는 것이 중요할 수 있다는 정도만 가정할 수 있습니다.

In [ ]:
y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)

tn = int(confusion_df.loc["actual_completed", "pred_completed"])
fp = int(confusion_df.loc["actual_completed", "pred_cancelled"])
fn = int(confusion_df.loc["actual_cancelled", "pred_completed"])
tp = int(confusion_df.loc["actual_cancelled", "pred_cancelled"])
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(confusion_df.values)
for i in range(confusion_df.shape[0]):
    for j in range(confusion_df.shape[1]):
        ax.text(j, i, int(confusion_df.iloc[i, j]), ha="center", va="center")
ax.set_xticks([0, 1], labels=["Pred Completed", "Pred Cancelled"])
ax.set_yticks([0, 1], labels=["Actual Completed", "Actual Cancelled"])
ax.set_title("Final Test Confusion Matrix")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "step06_confusion_matrix.png", dpi=150)
plt.show()

### 업무·분석적 의미와 한계

FP가 많으면 정상 완료 주문에 불필요한 확인·개입 비용이 발생할 수 있고, FN이 많으면 실제 취소 위험 주문을 놓칠 수 있습니다. 어느 오류를 더 줄일지는 **개입 비용, 취소 손실액, 고객 경험, 담당자 처리 용량** 같은 업무 정보가 추가로 필요합니다.

또한 이 모델이 발견한 것은 **예측 패턴**이지 취소의 원인이 아닙니다.

## 9. Internal / Public 결과 구분

오류 분석을 위한 내부 결과에는 source index 같은 추적 정보가 필요할 수 있지만, 공개 Prediction에서는 식별자를 제거합니다.

공개 컬럼:
`record_id, actual_is_cancelled, predicted_is_cancelled, cancel_probability, model, threshold`

In [ ]:
prediction_internal = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)
prediction_public = public_prediction_result(prediction_internal)

display(prediction_public.head())
print("Public columns:", prediction_public.columns.tolist())

forbidden_public = {"source_index", "order_id", "customer_id", "product_id"}
assert not forbidden_public.intersection(prediction_public.columns)
print("Privacy-safe Output: PASS")

## 10. Validation Evidence와 전체 재실행

자동 Validation PASS는 **코드 계약이 지켜졌는지**를 확인하는 Evidence입니다. 이것만으로 운영 적합성, 공정성, 시간에 따른 성능 변화까지 증명되는 것은 아닙니다.

In [ ]:
validation = build_classification_validation(
    model_data=model_data,
    features=features,
    merge_checks=merge_checks,
    y_train=y_train,
    y_valid=y_valid,
    y_test=y_test,
    validation_comparison=validation_comparison,
    selected_model_name=selected_model_name,
    threshold_df=threshold_df,
    selected_threshold=selected_threshold,
    prediction_result_public=prediction_public,
)
display(validation)
assert validation["status"].eq("PASS").all()
print("Validation Evidence: ALL PASS")

In [ ]:
result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print("재실행 선택 모델:", result["selected_model_name"])
print("재실행 Threshold:", result["selected_threshold"])
display(result["test_metrics"])
display(result["validation"])

assert result["selected_model_name"] == selected_model_name
assert np.isclose(result["selected_threshold"], selected_threshold)
assert result["validation"]["status"].eq("PASS").all()
print("전체 재실행: PASS")

## 11. 최종 사용 판단

**현재 판단: 추가 검증 후 사용 가능**

판단 근거:
1. Dummy baseline과 Logistic Regression / Random Forest를 같은 Validation 기준으로 비교했습니다.
2. 모델과 Threshold를 Validation에서 고정하고 Final Test는 마지막 평가에만 사용했습니다.
3. 누수·병합·공개 결과 개인정보 범위를 자동 Evidence로 검증했습니다.

### 현재 모델의 가장 큰 위험

현재 split은 교육용 random split이므로 미래 주문에 같은 성능이 유지된다고 보장할 수 없습니다. 또한 현재 feature가 취소의 원인을 설명하는 것도 아닙니다.

### 다음 개선 우선순위

1. 시간 순서 기반 out-of-time 평가
2. 실제 FP/FN 비용을 반영한 Threshold 정책
3. 기간·고객군별 성능과 공정성 점검

## 12. 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 함께 확인했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.

최종 제출 파일: `assignments/chapter10/chapter10.ipynb`